# Student Identification & Submission Details

Please complete the following information before submitting your notebook.

**UNI:**  lmv2141

**Full Name:**  Lindsay Varzarevsky

**Public GitHub Repository URL (Final Project Report & Code):** https://github.com/lindsayvarzz/AML_Project_1

---

## Submission Requirements

- The GitHub repository **must be public**.
- The repository should include:
  - A complete and well-formatted final report
  - Clean, well-documented source code
  - Clear instructions for reproducing results (if applicable)
- Verify that all links are functional and accessible prior to submission.

# Project Submission Guidelines

### Submission Due Date: March 9, 2026

Throughout this notebook, you will encounter the keyword **`TODO`** in multiple sections. Each `TODO` indicates a required component that must be completed by you.

These may include:

- Implementing missing code
- Specifying or tuning model parameters
- Writing explanations and technical reasoning
- Performing analysis and interpreting results
- Justifying modeling decisions
- Completing evaluation procedures

---

## Expectations

You are expected to:

- Replace **every** `TODO` with correct, fully functional code.
- Provide complete, clear, and technically sound explanations where required.
- Justify modeling choices using appropriate reasoning and evidence.
- Ensure the notebook executes from start to finish **without errors**.

> **Note:** Any section that still contains `TODO` at the time of submission will be considered **incomplete**.

# World Happiness Classification Competition

## Project Objectives

This competition is designed to help you:

- Develop a deep understanding of how machine learning models function
- Understand the role and impact of model hyperparameters
- Learn from systematic experimentation and model iteration
- Produce a professional, well-structured notebook report
- Publish the final project as a portfolio-ready GitHub repository

---

## Project Workflow

You will complete the project through the following stages:

1. **Load and Merge Datasets**  
   - Import all relevant datasets  
   - Perform necessary joins and validation  

2. **Data Preprocessing**  
   - Build preprocessing pipelines using `sklearn`'s `ColumnTransformer`  
   - Handle missing values, encoding, scaling, and feature engineering  

3. **Model Training**  
   - Fit the model using the processed data  
   - Save both the preprocessing pipeline and trained model  

4. **Prediction and Evaluation**  
   - Generate predictions  
   - Evaluate performance using appropriate metrics  

5. **Hyperparameter Tuning**  
   - Systematically tune model parameters  
   - Compare performance across configurations  

6. **Deep Learning Experimentation**  
   - Implement and evaluate a neural network approach  
   - Compare results with classical ML models  

7. **Model Explainability (SHAP)**  
   - Apply SHAP for feature importance analysis  
   - Interpret and explain model decisions  

8. **Final Report Submission**  
   - Clean and organize the notebook  
   - Ensure full reproducibility  
   - Upload the final report and code to a public GitHub repository  

---

## 0. Loading Datasets

In this section, we load the **World Happiness 2023** datasets that will be used throughout the project.

**Objectives of this stage:**

- Import the datasets into the notebook environment
- Inspect their structure and contents
- Verify column names and data types
- Identify potential inconsistencies before merging

> Careful dataset inspection at this stage prevents downstream errors and ensures a clean modeling pipeline.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

#Load the dataset
whr_df = pd.read_csv('data/WHR_2023.csv')

#Inspect the first few rows to understand the structure
whr_df

,country,region,happiness_score,gdp_per_capita,social_support,healthy_life_expectancy,freedom_to_make_life_choices,generosity,perceptions_of_corruption
0,Finland,Western Europe,7.804,1.888,1.585,0.535,0.772,0.126,0.535
1,Denmark,Western Europe,7.586,1.949,1.548,0.537,0.734,0.208,0.525
2,Iceland,Western Europe,7.530,1.926,1.620,0.559,0.738,0.250,0.187
3,Israel,Middle East and North Africa,7.473,1.833,1.521,0.577,0.569,0.124,0.158
4,Netherlands,Western Europe,7.403,1.942,1.488,0.545,0.672,0.251,0.394
...,...,...,...,...,...,...,...,...,...
132,Congo (Kinshasa),Sub-Saharan Africa,3.207,0.531,0.784,0.105,0.375,0.183,0.068
133,Zimbabwe,Sub-Saharan Africa,3.204,0.758,0.881,0.069,0.363,0.112,0.117
134,Sierra Leone,Sub-Saharan Africa,3.138,0.670,0.540,0.092,0.371,0.193,0.051
135,Lebanon,Middle East and North Africa,2.392,1.417,0.476,0.398,0.123,0.061,0.027


In [3]:
# Convert the regression target ('happiness_score') into classification labels
# We'll use quartiles to create 4 happiness categories: Very Low, Low, High, Very High

# Define quartiles
whr_df['happiness_category'] = pd.qcut(whr_df['happiness_score'],
                                       q=5,
                                       labels=['Very Low', 'Low','Average', 'High', 'Very High'])

# Select features and target
X = whr_df.drop(columns=['happiness_score', 'happiness_category'])
y = whr_df['happiness_category']

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Convert y_train and y_test to numerical labels
y_train_labels = y_train.astype('category').cat.codes

#TODO (Completed): Complete in a similar manner as above
y_test_labels = y_test.astype('category').cat.codes

### Conceptual Question

In the next cell, briefly explain the following:

1. What does `y_train.astype('category').cat.codes` do?
2. What is the difference between `y_train_labels` and `y_train`?

Keep your explanation clear and concise.

Response:

1. y_train.astype('category').cat.codes turns the target variable's labeled string categories that we defined earlier (ex: "very low","low","average","high","very high", etc) into numerically coded categories so that a machine can interpret it where, for example, 0 can mean very low, 1 = low, 2 = average, 3 = high, 4 = very high.

2. The difference between y_train_labels and y_train is that y_train_labels are the same categories, but just numerically encoded for machine learning readibility. It is best practice to keep the y_train dataset as strings for interpretability, but use the y_train_labels dataset to use for model training.

### Add New Data

In [4]:
#Truncated and cleaned up region data to merge
countrydata = pd.read_csv("data/newcountryvars.csv")

countrydata.head()

,country_name,population,population_below_poverty_line,hdi,life_expectancy,expected_years_of_schooling,mean_years_of_schooling,gni
0,India,1339180127,21.9,0.623559,68.322,11.696590,6.298834,5663.474799
1,Nigeria,190886311,70.0,0.527105,53.057,9.970482,6.000000,5442.901264
2,Mexico,129163276,46.2,0.761683,76.972,13.299090,8.554985,16383.106680
3,Pakistan,197015955,29.5,0.550354,66.365,8.106910,5.089460,5031.173074
4,Bangladesh,164669751,31.5,0.578824,71.985,10.178706,5.241577,3341.490722


In [5]:
# Merge in new data to X_train and X_test by taking "country" from first table and "country_name" from 2nd table.
# Also check which countries are common in both the datasets, and which type of merge will you perform for the best results.
# Hint: Look on the 'how' parameter of megre function of pandas.

#to make a complete modeling-ready dataset, it looks like there's some cleaning left to do, I should ensure that we are merging all same countries where applicable. *confirmed this at TA Office Hours*
# There are are a lot of countries that will not be able to merge because they have different punctuation/naming conventions in the different datasets. 

# So first, I'm going to identify the countries across the two datasets with these issues, standardize/normalize, manually define country mappings, and then merge them. This will be beneficial to having as complete a dataset as possible. 

#  code to check which countries are different between the datasets:

import re

def normalize_country_name(name):
    if pd.isna(name):
        return name
    name = str(name).strip().lower()
    name = re.sub(r"[^\w\s()\-&,]", "", name)
    name = re.sub(r"\s+", " ", name)
    return name

country_map = {
    "hong kong sar of china": "hong kong",
    "hong kong special administrative region": "hong kong",
    "turkiye": "turkey",
    "republic of turkiye": "turkey",
    "czechia": "czech republic",
    "viet nam": "vietnam",
    "taiwan province of china": "taiwan",
    "china, taiwan province of china": "taiwan",
    "taiwan, china": "taiwan",
    "state of palestine": "palestine",
    "congo (brazzaville)": "republic of the congo",
    "congo, republic of": "republic of the congo",
    "republic congo": "republic of the congo",
    "congo (kinshasa)": "democratic republic of the congo",
    "democraticrepublic of congo": "democratic republic of the congo",
    "democratic republic of congo": "democratic republic of the congo",
    "congo, democratic republic of the": "democratic republic of the congo",
    "drc": "democratic republic of the congo",
    "macedonia": "north macedonia",
    "former yugoslav republic of macedonia": "north macedonia",
    "swaziland": "eswatini"
}

countrydata["country_clean"] = (
    countrydata["country_name"]
    .apply(normalize_country_name)
    .replace(country_map)
)


#TODO:

X_train["country_clean"] = X_train["country"].apply(normalize_country_name).replace(country_map)
X_test["country_clean"] = X_test["country"].apply(normalize_country_name).replace(country_map)

countrydata_subset = countrydata[
    [
        "country_name",
        "country_clean",
        "population",
        "population_below_poverty_line",
        "hdi",
        "life_expectancy",
        "expected_years_of_schooling",
        "mean_years_of_schooling",
        "gni"
    ]
]

X_train = pd.merge(X_train, countrydata_subset, on="country_clean", how="left")
X_test = pd.merge(X_test, countrydata_subset, on="country_clean", how="left")

X_train = X_train.drop(columns=["country_name"])
X_test = X_test.drop(columns=["country_name"])

In [6]:
X_train

,country,region,gdp_per_capita,social_support,healthy_life_expectancy,freedom_to_make_life_choices,generosity,perceptions_of_corruption,country_clean,population,population_below_poverty_line,hdi,life_expectancy,expected_years_of_schooling,mean_years_of_schooling,gni
0,Madagascar,Sub-Saharan Africa,0.632,0.779,0.178,0.187,0.177,0.134,madagascar,2.557090e+07,70.7,0.512149,65.515,10.346140,6.145955,1319.699397
1,Mauritania,Sub-Saharan Africa,1.099,0.764,0.244,0.320,0.130,0.195,mauritania,4.420184e+06,31.0,0.513106,63.239,8.463790,4.266000,3527.264154
2,Mongolia,East Asia,1.379,1.494,0.244,0.425,0.239,0.058,mongolia,3.075647e+06,21.6,0.734832,69.806,14.845520,9.750000,10449.207270
3,Tajikistan,Commonwealth of Independent States,0.972,1.248,0.291,0.599,0.104,0.292,tajikistan,8.921343e+06,31.5,0.627472,69.582,11.261860,10.355820,2600.585607
4,Ukraine,Central and Eastern Europe,1.358,1.354,0.355,0.551,0.265,0.016,ukraine,4.422295e+07,24.1,0.743049,71.129,15.306720,11.340000,7361.011228
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,South Korea,East Asia,1.853,1.188,0.603,0.446,0.112,0.163,south korea,5.098221e+07,12.5,0.900992,82.128,16.587520,12.179211,34540.649270
91,Ethiopia,Sub-Saharan Africa,0.793,1.114,0.250,0.451,0.283,0.101,ethiopia,1.049574e+08,29.6,0.447750,64.602,8.351520,2.580780,1522.954782
92,China,East Asia,1.510,1.249,0.468,0.666,0.115,0.145,china,1.409517e+09,3.3,0.737681,75.963,13.535750,7.641840,13345.477460
93,Peru,Latin America and Caribbean,1.390,1.153,0.499,0.549,0.073,0.027,peru,3.216548e+07,22.7,0.739749,74.814,13.386340,9.013470,11294.840330


---

## 1. Exploratory Data Analysis and Visualization (EDAV)

In [ ]:
print(X_train.dtypes)

### Describe What You Observe

*Replace this section with your response.*

**TODO**

### Missing Values Analysis

Compute the number and percentage of missing values for each column and present the results in a clear summary table.

In [ ]:
# Your code here: TODO

### Distribution of Key Numerical Features

Select relevant numerical features and plot their histograms. Choose features that may influence the target variable, and briefly comment on notable patterns, skewness, or outliers.

In [ ]:
# Your plotting code here: TODO

### Distribution of Categorical Variables

Plot the distribution of relevant categorical variables using appropriate visualizations (e.g., count plots or bar charts). Briefly comment on class imbalance or notable patterns.

In [ ]:
# Your plotting code here: TODO

### Feature Correlation Analysis

Analyze the relationships between numerical features using correlation coefficients. Compute and visualize:

- **Pearson correlation** — linear relationships
- **Spearman correlation** — monotonic relationships
- **Kendall correlation** — rank-based relationships

Use correlation matrices and heatmaps where appropriate. Briefly interpret strong positive or negative correlations and discuss potential multicollinearity.

In [ ]:
# Your code here: TODO

### Bivariate Analysis and Relationship Exploration

Explore relationships between features and examine how they associate with the target variable. Include:

- Bivariate plots (e.g., scatter plots, box plots, grouped bar charts)
- Correlation tables
- Comparisons of feature distributions across target classes

Briefly summarize key relationships and any patterns that may influence model performance.

In [ ]:
# Your plotting code(s) here: TODO

### Outlier Detection

Identify potential outliers in the numerical features using appropriate methods such as:

- Box plots
- Z-score analysis
- Interquartile Range (IQR) method

Highlight any significant anomalies and briefly comment on whether they should be retained, transformed, or removed.

In [ ]:
# Your code here: TODO

### Observations and General Comments

Summarize your key findings from the exploratory analysis. Include:

- Important patterns or relationships identified
- Presence of missing values or outliers
- Potential feature engineering considerations
- Any preprocessing steps that should be applied before modeling

Keep your comments concise and evidence-based.

*Replace this section with your response.*

**TODO**

---

## 2. Feature Engineering

Apply log transformations to normalize skewed numerical features and improve model stability (if applicable).

In [ ]:
# Your code here: TODO

Create at least one interaction feature to capture relationships between existing variables and enhance predictive power.

In [ ]:
# Your code here: TODO

---

## 3. Data Preprocessing

Use `sklearn`'s `ColumnTransformer` to preprocess the data. Write a preprocessing function and save the fitted preprocessor for later use.

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Create the preprocessing pipelines for both numeric and categorical data.

numeric_features = ## Drop all the non-numerical features from X_train: TODO
numeric_features=numeric_features.columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)), ## Is this good enough?
    ('scaler', StandardScaler())]) # You will need to describe why this is being done in the next cell

categorical_features = ['region', 'sub-region']

#Replacing missing values with Modal value and then one hot encoding.
categorical_transformer = Pipeline(steps=[
    ('imputer',  ## Fill here: TODO)),
    ('onehot', OneHotEncoder(handle_unknown= ## Fill here: TODO))])

# final preprocessor object set up with ColumnTransformer
preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features),('cat', categorical_transformer, categorical_features)])

#Fit your preprocessor object
preprocess=preprocessor.fit(X_train)

### Explanation

Briefly explain what the preprocessing steps above do and why they are needed.

If you changed the imputation strategy (or any other preprocessing choice), please state:

- What you changed
- Why you made that change

*Replace this section with your response.*

**TODO**

In [ ]:
# function to transform data with preprocessor

def preprocessor(data):
    data.drop(['country', 'region'], axis=1)
    preprocessed_data=preprocess.transform(data)
    return preprocessed_data

### Conceptual Question

Explain the differences between the following:

- The `preprocessor` **object**
- The `preprocess` **object**
- The `preprocessor` **function**
- The final `preprocessed_data` **returned**

Clearly distinguish between objects, functions, and transformed data.

*Replace this section with your response.*

**TODO**

In [ ]:
# check shape of X data after preprocessing it using our new function
preprocessor(X_train).shape

---

## 4. Model Training and Saving Artifacts

Fit the model on the preprocessed training data. Save the fitted preprocessor and the trained model for reuse (e.g., inference and submission).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = ## Define a Random Forest Model here, fit it, and score it: TODO

# Your cell should have a score between 0-1 as output
# add your lines of code here: TODO

---

## 5. Model Evaluation

Generate predictions using `X_test`. Compare the predictions with the true labels in `y_test` using appropriate evaluation metrics.

In [ ]:
#-- Generate predicted values (Model 1)
prediction_labels = model.predict(preprocessor(X_test))

# Write lines of code to show model performance by comparing prediction_labels with true labels: TODO


---

## 6. Hyperparameter Experimentation

Repeat the training and evaluation process with different model parameters to improve performance. Track results across experiments and report the best-performing configuration.

In [ ]:
# Train model 2 using same preprocessor (note that you could save a new preprocessor, but we will use the same one for this example).
from sklearn.ensemble import RandomForestClassifier

model_2 = ## Make a new model with changed parameters to improve the score: TODO
# write your lines of code here TODO

### Reflection on Hyperparameter Changes

Briefly explain:

- What parameters you changed
- What each parameter controls
- Why the changes improved (or did not improve) performance

Support your explanation with evidence from your results.

*Replace this section with your response.*

**TODO**

In [ ]:
#Evaluate Model 2:

#-- Generate predicted y values (Model 2)
prediction_labels = # Predict : TODO

# Write lines of code below to show model performance by comparing prediction_labels with true labels: TODO


### Discussion Question

Do you think it is worth making more changes to the parameters? Should we keep trying random values and see what works better? What is an alternative to doing this manually?

*Replace this section with your response.*

**TODO**

In [ ]:
# Submit a third model using GridSearchCV

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import numpy as np

param_grid = # Use np.arange to create a sequence of numbers for each parameter's space you think should be searched: TODO

# Create a RandomForestClassifier object TODO

gridmodel = # Read GridSearchCV docs and create an object with RandomForestClassifier as the model TODO

#Fit the GridSearchCV object to the preprocessed training data: TODO


#extract best score and parameter by calling objects "best_score_" and "best_params_"
print("best mean cross-validation score: {:.3f}".format(gridmodel.best_score_))
print("best parameters: {}".format(gridmodel.best_params_))

# Make predictions using the best model : TODO


In [ ]:
#Submit Model 3:

#-- Generate predicted values


## Write lines of code to show model performance by comparing prediction_labels with true labels: TODO


In [ ]:
# Here are several classic ML architectures you can consider choosing from to experiment with next:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import GradientBoostingClassifier


model = ## Read documentations of imported models and fit them. TODO

#-- Generate predicted values
prediction_labels = model.predict(preprocessor(X_test))

## Write lines of code to show model performance by comparing prediction_labels with true labels: TODO


### Discussion Question

Describe the parameters you defined in `GradientBoostingClassifier`, and/or `BaggingClassifier`, and/or `KNN`, and/or `SVC`. What worked and why?

*Replace this section with your response.*

**TODO**

---

## 7. Basic Deep Learning

In [ ]:
# Now experiment with deep learning models:
import keras
from keras.models import Sequential
from keras.layers import Dense, Activation

# Preprocess input features: TODO

feature_count=#count features in input data: TODO

keras_model = ## Define a Neural Network Model with 5 layers 128->64->64->32->(?) TODO


#Use Softmax activation in last layer. How many neurons should there be in the last layer?

# Encode categorical labels : TODO


# Compile model
keras_model.compile(loss='categorical_crossentropy', optimizer='sgd', metrics=['accuracy'])

# Fitting the NN to the Training set
keras_model.fit(preprocessor(X_train), y_train, ## Note that keras models require a one-hot-encoded y_train object
               batch_size = 20,
               epochs = 300, validation_split=0.25)

### Discussion Question

Which activation functions did you use in the hidden layers? Why was **softmax** used in the final layer?

*Replace this section with your response.*

**TODO**

### Discussion Question

Was it a good idea to train for 300 epochs? Should you train longer? Why or why not?

*Replace this section with your response.*

**TODO**

### Discussion Question

Why was `loss='categorical_crossentropy'` and `optimizer='sgd'` used? Would you change anything? Why or why not?

*Replace this section with your response.*

**TODO**

### Plot Training Curves

Extract the model training history and plot the learning curves (e.g., training vs. validation loss and accuracy) across epochs.

In [ ]:
## Your code to plot training and validation curves in a single plot (Make changes in the model cell to be able to do this) : TODO

In [ ]:
#-- Generate predicted y values

#Note: Keras predict returns the predicted column index location for classification models
prediction_column_index= # Predict TODO

# extract correct prediction labels
prediction_labels = [y_train.columns[i] for i in prediction_column_index]

## Write lines of code to show model performance by comparing prediction_labels with true labels: TODO


### Regularization: Dropout and Batch Normalization

Implement regularization techniques such as **Dropout** and **Batch Normalization** to improve generalization and observe the change in performance.

> **Note:** Compare training vs. test (or validation) loss and accuracy before and after adding regularization.

In [ ]:
# Write your lines of code here: TODO

*Replace this section with your response.*

**TODO**

### Activation Function Experimentation

Experiment with different activation functions — **ReLU**, **LeakyReLU**, **Tanh**, and **Sigmoid** — and compare their impact on model performance.

In [ ]:
# Your lines of code here: TODO

---

## 8. Explainability – SHAP Feature Importance

To better understand the model's predictions, we will use **SHAP (SHapley Additive exPlanations)** to analyze feature importance.

**How SHAP works:**

- SHAP assigns each feature a contribution score for every prediction.
- It uses Shapley values (from cooperative game theory) to fairly distribute importance across features.

We will apply SHAP to visualize and interpret the model's feature contributions.

In [ ]:
# Import necessary libraries
import shap
import matplotlib.pyplot as plt

# Initialize SHAP explainer: TODO
# Define an explainer that will help us interpret the model's decisions
# (Hint: Use shap.Explainer with the trained model and X_test data)

explainer = ## Initialize SHAP explainer using the trained model: TODO

# Compute SHAP values for X_test
# This step generates Shapley values, which explain how each feature contributes to predictions
shap_values = ## Apply the explainer to X_test: TODO

## Write lines of code here to transform the test data using the preprocessor : TODO
##and Generate generic feature names matching the number of columns after preprocessing : TODO

# Generate SHAP summary plot
# This plot will show which features have the most impact on predictions
shap.summary_plot(## Pass the required parameters to create a summary plot: TODO)

# Your cell should output a SHAP summary plot showing the most important features.


### Experimentation

In [ ]:
## You are encouraged to try more experimentation and any other models by adding more code cells to this notebook:

## You can also try to import any new dataset pertaining to countries, merge it, and see if it helps the predictions.
## If it does not, try to explain why it wasn't helpful by exploring variable relationships.

Deep learning models are often considered "black boxes" due to their complexity. Use SHAP to explain your model's predictions.

After applying SHAP, discuss:

- Does it provide a clear and sufficient explanation of how the model makes decisions?
- How easy or difficult is it to justify your model's predictions using this technique?

In [ ]:
## Your lines of Code and Answer: TODO

*Replace this section with your response.*

**TODO**

---

## 9. Final Report Submission (GitHub)

This is your final project to showcase on GitHub.

### Instructions

1. Create a new notebook for the final report.
2. Include relevant visualizations.
3. Reproduce the code for the best-performing model(s) and display results.
4. Summarize key insights and observed behaviors.
5. Present the work in a clean, concise report format (within the `.ipynb`).
6. Upload the final notebook to a new repository on your personal GitHub account.
7. Paste the link to your final repository at the top of this notebook where requested.